In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [2]:
claims1 = pd.read_csv("Medicare_Claims_data_part_1.csv", dtype=str)
claims2 = pd.read_csv("Medicare_Claims_data_part_2.csv", dtype=str)
claims3 = pd.read_csv("Medicare_Claims_data_part_3.csv", dtype=str)
claims4 = pd.read_csv("Medicare_Claims_data_part_4.csv", dtype=str)
claims5 = pd.read_csv("Medicare_Claims_data_part_5.csv", dtype=str)

hcp_raw_df = pd.read_csv("HCP_demographics_data.csv", dtype=str)
patient_raw_df = pd.read_csv("Patient_demographics_data.csv", dtype=str)
territory_raw_df = pd.read_csv("Zip_to_Territory_Mapping.csv", dtype=str)
diagnosis_raw_df = pd.read_csv("Diagnosis_Code_Mapping.csv", dtype=str)

FileNotFoundError: [Errno 2] No such file or directory: 'Medicare_Claims_data_part_1.csv'

In [ ]:
claims_df = pd.concat([claims1, claims2, claims3, claims4, claims5], ignore_index=True)
print("Total rows:", claims_df.shape)

In [ ]:
claims_df['npi_id'] = claims_df['fac_prvdr_npi_num'].str.split().str[-1]
claims_df['npi_id'] = claims_df['npi_id'].str.replace('.0', '', regex=False)

In [ ]:
claims_df = claims_df.rename(columns={
    'cur_clm_uniq_id': 'claim_id',
    'bene_mbi_id': 'patient_id',
    'clm_from_dt': 'claim_date',
    'clm_dgns_cd': 'diagnosis_code',
    'clm_line_hcpcs_cd': 'procedure_code'
})

In [ ]:
claims_df['claim_date'] = pd.to_datetime(claims_df['claim_date'], errors='coerce')
claims_df['diag_letter'] = claims_df['diagnosis_code'].str[0]

In [ ]:
target_codes = ['J1885', 'J2250', 'J2704', 'J3010']

target_claim_ids = claims_df[
    claims_df['procedure_code'].isin(target_codes)
]['claim_id'].unique()

claims_filtered_df = claims_df[
    claims_df['claim_id'].isin(target_claim_ids)
]

print("Filtered rows:", claims_filtered_df.shape)

In [ ]:
hcp_df = hcp_raw_df.copy()
hcp_df.columns = ['npi_id', 'address', 'hcp_city', 'hcp_state', 'hcp_zip', 'hcp_specialty']

hcp_df['npi_id'] = hcp_df['npi_id'].str.strip()
hcp_df['hcp_zip'] = hcp_df['hcp_zip'].str.strip().str.zfill(5)

In [ ]:
patient_df = patient_raw_df.copy()
patient_df.columns = ['patient_id', 'patient_age', 'patient_gender']

In [ ]:
territory_df = territory_raw_df.copy()
territory_df.columns = ['hcp_zip', 'territory', 'region']

territory_df['hcp_zip'] = territory_df['hcp_zip'].str.strip().str.zfill(5)

In [ ]:
diagnosis_df = diagnosis_raw_df.copy()
diagnosis_df.columns = ['diag_letter', 'diagnosis_specialty']

diagnosis_df['diag_letter'] = diagnosis_df['diag_letter'].str.strip().str.upper()

In [ ]:
master_df = claims_filtered_df.copy()

# Join HCP
master_df = master_df.merge(
    hcp_df[['npi_id', 'hcp_zip', 'hcp_specialty', 'hcp_state', 'hcp_city']],
    on='npi_id',
    how='left'
)

# Join Territory
master_df = master_df.merge(
    territory_df,
    on='hcp_zip',
    how='left'
)

# Join Patient
master_df = master_df.merge(
    patient_df,
    on='patient_id',
    how='left'
)

# Join Diagnosis
master_df = master_df.merge(
    diagnosis_df,
    on='diag_letter',
    how='left'
)

print(master_df.shape)

In [ ]:
target_codes = ['J1885', 'J2250', 'J2704', 'J3010']

master_df['product_code'] = np.where(
    master_df['procedure_code'].isin(target_codes),
    master_df['procedure_code'],
    np.nan
)

master_df = master_df[master_df['product_code'].notna()]

print(master_df['product_code'].unique())

In [ ]:
product_map = {
    'J1885': ('Ketotrom', 'Our Old Brand'),
    'J2250': ('Midoride', 'Our New Brand'),
    'J3010': ('Fentirate', 'Competitor'),
    'J2704': ('Profativ', 'Alternative Competitor')
}

master_df['product_name'] = master_df['product_code'].map(lambda x: product_map[x][0])
master_df['product_role'] = master_df['product_code'].map(lambda x: product_map[x][1])

In [ ]:
# Row ID
master_df['row_id'] = range(1, len(master_df) + 1)

# Rename claim_id
master_df = master_df.rename(columns={'claim_id': 'source_claim_id'})

# Year
master_df['claim_year'] = master_df['claim_date'].dt.year

In [ ]:
master_df['patient_age'] = pd.to_numeric(master_df['patient_age'], errors='coerce')

bins = [0,30,40,50,60,70,80,100]
labels = ['18-30','31-40','41-50','51-60','61-70','71-80','81+']

master_df['age_group'] = pd.cut(master_df['patient_age'], bins=bins, labels=labels)

In [ ]:
first_year = master_df.groupby(['npi_id', 'product_code'])['claim_year'].min().reset_index()
first_year = first_year.rename(columns={'claim_year': 'first_year'})

master_df = master_df.merge(first_year, on=['npi_id', 'product_code'], how='left')

master_df['writer_type'] = np.where(
    master_df['claim_year'] == master_df['first_year'],
    'New Writer',
    'Continuing Writer'
)

In [ ]:
master_df['total_charge'] = pd.to_numeric(master_df.get('clm_mdcr_instnl_tot_chrg_amt'), errors='coerce')
master_df['claim_payment'] = pd.to_numeric(master_df.get('clm_pymt_amt'), errors='coerce')

master_df['is_adjustment'] = master_df['total_charge'] < 0

In [ ]:
master_df = master_df[[
    'row_id',
    'source_claim_id',
    'patient_id',
    'npi_id',
    'product_code',
    'product_name',
    'product_role',
    'claim_year',
    'claim_date',
    'diagnosis_code',
    'diag_letter',
    'diagnosis_specialty',
    'claim_payment',
    'total_charge',
    'is_adjustment',
    'hcp_specialty',
    'hcp_state',
    'hcp_city',
    'hcp_zip',
    'territory',
    'region',
    'patient_age',
    'patient_gender',
    'age_group',
    'writer_type'
]]

In [ ]:
master_df.head()

In [ ]:
print(master_df['product_code'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

plt.rcParams.update({
    'figure.figsize': (12,7),
    'axes.titlesize': 16,
    'axes.labelsize': 13
})

color_map = {
    'Ketotrom': '#6C8EBF',
    'Midoride': '#E27D60',
    'Fentirate': '#2A9D8F',
    'Profativ': '#B5838D'
}

In [ ]:
claims = master_df.groupby(['claim_year','product_name'])['source_claim_id'].nunique().reset_index()
claims['share'] = claims.groupby('claim_year')['source_claim_id'].transform(lambda x: x/x.sum())

pivot = claims.pivot(index='claim_year', columns='product_name', values='share')

# --- FIX 1: fill missing values (prevents plotting issues)
pivot = pivot.fillna(0)

# --- FIX 2: plot
ax = pivot.plot(
    kind='bar',
    stacked=True,
    color=[color_map.get(col, '#333333') for col in pivot.columns],
    figsize=(10,6)
)

# --- FIX 3: move legend OUTSIDE (no overlap)
ax.legend(title="Product", bbox_to_anchor=(1.02, 1), loc='upper left')

# --- FIX 4: clean labels
plt.title("Market Share by Claims", fontweight='bold')
plt.ylabel("Share")
plt.xlabel("Year")

# --- FIX 5: adjust spacing (instead of tight_layout)
plt.subplots_adjust(right=0.8)

plt.show()

Fentirate shows a consistent upward trend in market share across all years, indicating strong competitive momentum. Midoride remains relatively flat, suggesting it is struggling to gain traction. Ketotrom shows a clear decline, indicating loss of relevance in the market.

In [ ]:
def beautify(ax, title):
    ax.set_title(title, fontweight='bold', pad=12)
    ax.legend(title="Product", bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.subplots_adjust(right=0.8)

In [ ]:
patients_pivot = patients.pivot(index='claim_year', columns='product_name', values='share')

patients_pivot.plot(kind='area', stacked=True, color=[color_map.get(col) for col in patients_pivot.columns], alpha=0.8)

ax = plt.gca()
beautify(ax, "Patient Share Trend")

plt.ylabel("Share")
plt.xlabel("Year")
plt.show()

Fentirate is steadily capturing a larger portion of patients over time, indicating growing patient-level adoption. Midoride’s patient share remains stagnant, suggesting limited penetration. This trend confirms that competitor growth is not just from doctors but also from broader patient usage.

In [ ]:
# Step 1: Create HCP data
hcp = master_df.groupby(['claim_year','product_name'])['npi_id'].nunique().reset_index()

hcp['share'] = hcp.groupby('claim_year')['npi_id'].transform(lambda x: x/x.sum())

# Step 2: Pivot
hcp_pivot = hcp.pivot(index='claim_year', columns='product_name', values='share')

# Step 3: Plot
hcp_pivot.plot(kind='line', marker='o', linewidth=3,
               color=[color_map.get(col) for col in hcp_pivot.columns])

ax = plt.gca()
beautify(ax, "HCP Share Trend")

plt.ylabel("Share")
plt.xlabel("Year")
plt.grid(alpha=0.3)

plt.show()

Fentirate shows a clear increase in the number of prescribing HCPs, indicating successful physician adoption. Midoride is not attracting new prescribers at the same rate. This suggests that competitor growth is driven by expanding doctor base rather than just higher usage.

In [ ]:
# Step 1: Create data
agg = master_df.groupby(['claim_year','product_name']).agg({
    'source_claim_id':'nunique',
    'npi_id':'nunique'
}).reset_index()

agg['claims_per_hcp'] = agg['source_claim_id'] / agg['npi_id']

# Step 2: Plot
sns.lineplot(
    data=agg,
    x='claim_year',
    y='claims_per_hcp',
    hue='product_name',
    palette=color_map,
    marker='o',
    linewidth=3
)

ax = plt.gca()
beautify(ax, "Claims per HCP (Doctor Productivity)")

plt.ylabel("Claims per HCP")
plt.xlabel("Year")
plt.grid(alpha=0.3)

plt.show()

Fentirate maintains a higher number of claims per HCP, indicating stronger physician preference and repeat usage. Midoride shows lower productivity per doctor, suggesting weaker engagement. This highlights a gap in physician confidence or satisfaction.

In [ ]:
territory_df = master_df.groupby(['territory','claim_year','product_name'])['source_claim_id'].nunique().reset_index()

In [ ]:
pivot = territory_df.pivot_table(index='territory',
                                 columns=['claim_year','product_name'],
                                 values='source_claim_id')

# Calculate YoY change for Midoride
pivot['YoY'] = (pivot[(2018,'Midoride')] - pivot[(2017,'Midoride')]) / pivot[(2017,'Midoride')]

top5 = pivot.sort_values('YoY').head(5).index

In [ ]:
plot_data = master_df[
    (master_df['territory'].isin(top5)) &
    (master_df['product_name'].isin(['Midoride','Fentirate']))
]

In [ ]:
plot_agg = plot_data.groupby(['territory','claim_year','product_name'])['source_claim_id'].nunique().reset_index()

In [ ]:
sns.barplot(
    data=plot_agg,
    x='territory',
    y='source_claim_id',
    hue='product_name',
    palette={'Midoride': '#E27D60', 'Fentirate': '#2A9D8F'},
    errorbar=None   # 🔥 THIS REMOVES THOSE LINES
)

ax = plt.gca()
beautify(ax, "Top 5 Territories: Competitive Shift")

plt.xticks(rotation=30)
plt.ylabel("Claims")
plt.xlabel("Territory")

plt.show()

The top 5 territories show a clear decline in Midoride claims between 2017 and 2018, while Fentirate shows strong growth in the same regions. This indicates direct competitive substitution rather than overall market decline. These territories represent high-priority areas where targeted sales and marketing efforts can help recover lost market share.

In [ ]:
diag = master_df['diagnosis_specialty'].value_counts().head(5)

diag.plot(
    kind='pie',
    autopct='%1.1f%%',
    wedgeprops=dict(width=0.4)
)

plt.title("Top Diagnosis Specialties", fontweight='bold')
plt.ylabel("")

plt.show()

A small number of diagnosis specialties account for a large proportion of claims, indicating that demand is concentrated in specific medical conditions. This suggests that targeting these key diagnosis segments can significantly improve market penetration. Focusing marketing efforts on these areas could drive higher adoption.

In [ ]:
hcp_spec = master_df.groupby('hcp_specialty')['npi_id'].nunique().sort_values(ascending=False)

hcp_spec.head(10).plot(kind='barh', color='#6C8EBF')

plt.title("Top HCP Specialties", fontweight='bold')
plt.xlabel("Number of HCPs")
plt.gca().invert_yaxis()

plt.show()

Anesthesiology is the dominant specialty prescribing these products, indicating that most demand originates from this group. This highlights a strong opportunity to focus sales and marketing efforts on anesthesiologists. Targeting high-volume specialties can significantly improve adoption.

In [ ]:
age = master_df['age_group'].value_counts().sort_index()

age.plot(kind='bar', color='#6C8EBF')

plt.title("Patient Age Distribution", fontweight='bold')
plt.xlabel("Age Group")
plt.ylabel("Count")

for i, v in enumerate(age.values):
    plt.text(i, v, str(v), ha='center')

plt.show()

Older age groups account for the majority of patients, indicating that demand is heavily concentrated among elderly populations. This suggests that treatment usage is linked to age-related conditions. Targeting messaging toward older patients can improve effectiveness.

In [ ]:
# Step 1: Sort data
df = master_df.sort_values(by=['npi_id','product_name','claim_year'])

# Step 2: Find first year each doctor used each product
df['first_year'] = df.groupby(['npi_id','product_name'])['claim_year'].transform('min')

# Step 3: Mark new writers
df['is_new_writer'] = df['claim_year'] == df['first_year']

# Step 4: Count new writers per year per product
new_agg = df[df['is_new_writer']].groupby(['claim_year','product_name'])['npi_id'].nunique().reset_index()

In [ ]:
sns.barplot(
    data=new_agg,
    x='claim_year',
    y='npi_id',
    hue='product_name',
    palette=color_map,
    errorbar=None
)

ax = plt.gca()
beautify(ax, "New Writer Growth (Corrected)")

plt.ylabel("Number of New HCPs")
plt.xlabel("Year")

plt.show()

The number of new writers is highest in 2016 for all products, followed by a decline in subsequent years. Fentirate does not show consistent growth in new writers but still maintains relatively higher acquisition compared to some competitors in later years. This suggests that while initial adoption was strong, sustaining new HCP onboarding remains a challenge across the market.

KBQ 3 — STRATEGIES TO STOP MARKET SHARE LOSS
1. OVERALL PROBLEM STATEMENT

The analysis shows that Midoride is losing market share to Fentirate due to weaker physician adoption, lower claims per HCP, and declining performance in key territories. While initial adoption peaked around 2016, Midoride has failed to sustain momentum in acquiring new prescribers and expanding usage among existing ones. In contrast, Fentirate demonstrates stronger competitive positioning across multiple dimensions.

2. KEY STRATEGIC RECOMMENDATIONS
1. Territory-Focused Recovery Strategy

The territory analysis shows that the largest declines in Midoride claims occur in specific high-impact regions where Fentirate is simultaneously growing. This indicates direct competitive displacement rather than overall market shrinkage.

Action:

Reallocate sales reps to top 5 declining territories
Increase frequency of HCP visits in these regions
Prioritize high-volume prescribers in these areas

Impact:
This targeted approach can quickly recover lost market share in the most critical regions.

 2. HCP Targeting and Specialty Focus

The data shows that anesthesiology dominates prescribing behavior, making it the most influential specialty in this market. However, Midoride is underperforming in attracting and retaining these high-value HCPs.

 Action:

Focus sales efforts on anesthesiologists and high-volume HCPs
Identify HCPs who switched to Fentirate and re-engage them
Provide targeted clinical messaging tailored to specialty needs

 Impact:
Improved penetration in key specialties will drive higher adoption and prescription volume.

 3. New Writer Acquisition Strategy

The new writer analysis shows that adoption peaked in 2016 and declined afterward, indicating market saturation and reduced onboarding of new prescribers. Midoride is particularly weak in sustaining new HCP growth.

 Action:

Launch targeted new writer programs (samples, trials, onboarding support)
Provide incentives for first-time prescriptions
Conduct awareness campaigns for untapped HCP segments

 Impact:
Increasing new writer acquisition is critical for long-term growth and competitive positioning.

 4. Improve HCP Productivity (Claims per HCP)

Fentirate shows higher claims per HCP, meaning doctors prescribe it more frequently once adopted. Midoride lags in usage intensity among existing prescribers.

 Action:

Strengthen engagement with existing Midoride users
Provide clinical evidence and support to increase prescription confidence
Use follow-ups and education to encourage repeat usage

 Impact:
Improving usage per HCP will increase overall claim volume without needing new doctors.

 5. Patient Segment Targeting Strategy

The age distribution analysis shows that elderly patients dominate the market, indicating that demand is concentrated in older populations.

 Action:

Position Midoride as a preferred option for elderly patients
Tailor messaging around safety and effectiveness for older age groups
Focus on treatment areas common in aging populations

 Impact:
Better alignment with dominant patient segments can improve adoption and outcomes.

 6. Promotional Strategy (Personal + Digital)

The declining trends indicate gaps in both physician engagement and awareness.

 Action:

Increase sales rep interactions in high-priority territories
Expand digital campaigns (emails, webinars, educational content)
Use hybrid promotion (rep + digital) for better reach

 Impact:
Improved visibility and engagement will help counter competitor growth.

 FINAL KBQ 3 SUMMARY

Midoride’s decline is driven by competitive displacement, weak new writer growth, and lower physician engagement. A focused strategy targeting key territories, high-value HCPs, and patient segments, combined with improved promotional efforts, can help stabilize and recover market share.

KBQ 4 — DATA EXPLORATION & GAPS
1. LIMITATIONS OF CURRENT DATA

The current analysis is based on Medicare claims data, which provides insights into prescription behavior but lacks visibility into the underlying drivers of these trends. While claims data shows what is happening, it does not fully explain why it is happening.

 2. KEY DATA GAPS & OPPORTUNITIES
 1. DDD (Drug Distribution Data)

The dataset does not include information on drug supply or inventory levels across hospitals and pharmacies.

 Why it matters:
Fentirate’s growth may be driven by higher availability rather than physician preference.

 Recommendation:
Integrate DDD data to understand supply-side dynamics and ensure adequate distribution of Midoride.

 2. Xponent (Prescriber-Level Data)

The current data does not track detailed prescribing behavior at the individual HCP level.

 Why it matters:
We cannot identify exactly which doctors switched from Midoride to Fentirate.

 Recommendation:
Use Xponent data to track switching behavior and target specific HCPs for retention strategies.

 3. Sales Force Activity Data

There is no visibility into sales rep activity across territories.

 Why it matters:
Declining territories may be due to reduced sales coverage rather than product performance.

 Recommendation:
Analyze rep call data to identify gaps in field force coverage and optimize resource allocation.

 4. NPP (Non-Personal Promotion Data)

Digital engagement data (emails, webinars, etc.) is not available.

 Why it matters:
We cannot measure the effectiveness of marketing campaigns.

 Recommendation:
Integrate digital engagement metrics to evaluate and optimize promotional strategies.

 5. Formulary / Payer Data

The dataset does not include payer or hospital formulary information.

 Why it matters:
Fentirate may have better insurance coverage or hospital approval, influencing prescribing behavior.

 Recommendation:
Analyze formulary data to identify access barriers and improve payer strategy.

 6. Broader Market Data (NPA/NSP)

The analysis is limited to Medicare patients only.

 Why it matters:
Trends may differ in commercial or private insurance populations.

 Recommendation:
Incorporate broader prescription audit data to validate findings across the entire market.

 FINAL KBQ 4 SUMMARY

While claims data provides strong insights into market trends, it lacks visibility into supply, physician behavior, and promotional effectiveness. Integrating additional datasets such as DDD, Xponent, and sales activity data would enable a more comprehensive understanding of market dynamics and support more precise strategic decisions.